In [1]:
%pip install ipython-sql --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

For Pandas

In [3]:
df = pd.read_csv("AAPL_data.csv")
df = df.sort_values("Date").reset_index(drop = True)

In [4]:
df.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,MA_20,STD_20,Upper_Band,Lower_Band,RSI,MACD,MACD_signal,MACD_diff
0,2016-09-12 00:00:00-04:00,23.474584,24.176649,23.447141,24.112617,181171200,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-09-13 00:00:00-04:00,24.585996,24.878714,24.524250,24.686617,248704800,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-09-14 00:00:00-04:00,24.864989,25.848336,24.835259,25.560192,443554800,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-09-15 00:00:00-04:00,26.038149,26.465792,25.953535,26.429201,359934400,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-09-16 00:00:00-04:00,26.326290,26.557261,26.079309,26.280552,319547600,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


For numpy

In [5]:
dates  = df["Date"].values
Open   = df["Open"].to_numpy(float)
High   = df["High"].to_numpy(float)
Low    = df["Low"].to_numpy(float)
Close  = df["Close"].to_numpy(float)
Volume = df["Volume"].to_numpy(float)
Divs   = df["Dividends"].to_numpy(float)
Splits = df["Stock Splits"].to_numpy(float)
MA20   = df["MA_20"].to_numpy(float)
STD20  = df["STD_20"].to_numpy(float)
UpperB = df["Upper_Band"].to_numpy(float)
LowerB = df["Lower_Band"].to_numpy(float)
RSI    = df["RSI"].to_numpy(float)
MACD   = df["MACD"].to_numpy(float)
MACDs  = df["MACD_signal"].to_numpy(float)
MACDd  = df["MACD_diff"].to_numpy(float)

n = len(Close)

DailyReturn = np.concatenate([[np.nan], np.diff(Close) / Close[:-1]])
Range       = High - Low
RunningMax  = np.maximum.accumulate(Close)
Drawdown    = Close / RunningMax - 1

For SQL

In [6]:
%load_ext sql

In [7]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [8]:
import math
import sqlite3

# ---------- Statistical: central tendency ----------

class Median:
    """MEDIAN(x) — sample median, interpolated for even n."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        n = len(self.v)
        if n == 0: return None
        s = sorted(self.v)
        return s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2


class Mode:
    """MODE(x) — most frequent value."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        if not self.v: return None
        from collections import Counter
        return Counter(self.v).most_common(1)[0][0]


# ---------- Statistical: dispersion ----------

class Variance:
    """VARIANCE(x) — sample variance (ddof=1)."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        n = len(self.v)
        if n < 2: return None
        m = sum(self.v) / n
        return sum((x - m) ** 2 for x in self.v) / (n - 1)


class VarPop:
    """VAR_POP(x) — population variance (ddof=0)."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        n = len(self.v)
        if n == 0: return None
        m = sum(self.v) / n
        return sum((x - m) ** 2 for x in self.v) / n


class StDev:
    """STDEV(x) — sample standard deviation (ddof=1)."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        n = len(self.v)
        if n < 2: return None
        m = sum(self.v) / n
        return math.sqrt(sum((x - m) ** 2 for x in self.v) / (n - 1))


class StdDevPop:
    """STDDEV_POP(x) — population standard deviation (ddof=0)."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        n = len(self.v)
        if n == 0: return None
        m = sum(self.v) / n
        return math.sqrt(sum((x - m) ** 2 for x in self.v) / n)


class MAD:
    """MAD(x) — median absolute deviation."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        n = len(self.v)
        if n == 0: return None
        s = sorted(self.v)
        med = s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2
        abs_dev = sorted(abs(x - med) for x in s)
        return abs_dev[n // 2] if n % 2 else (abs_dev[n // 2 - 1] + abs_dev[n // 2]) / 2


class Range:
    """RANGE(x) — max - min."""
    def __init__(self): self.mn = None; self.mx = None
    def step(self, x):
        if x is not None:
            self.mn = x if self.mn is None else min(self.mn, x)
            self.mx = x if self.mx is None else max(self.mx, x)
    def finalize(self):
        return None if self.mn is None else self.mx - self.mn


# ---------- Statistical: percentiles ----------

class Percentile:
    """PERCENTILE_CONT(p, x) — linear interpolation (Postgres-style).
    
    Note: SQLite passes BOTH arguments to __init__ and step on every call.
    """
    def __init__(self, *args):
        # SQLite passes the constant p as the first arg
        p = args[0] if args else None
        if p is None or not (0 <= p <= 1):
            raise ValueError("p must be between 0 and 1")
        self.p = float(p)
        self.v = []

    def step(self, *args):
        # args == (p, x) each call; we only accumulate x
        x = args[-1]
        if x is not None:
            self.v.append(x)

    def finalize(self):
        n = len(self.v)
        if n == 0:
            return None
        if n == 1:
            return self.v[0]
        s = sorted(self.v)
        idx = self.p * (n - 1)
        lo = int(math.floor(idx))
        hi = int(math.ceil(idx))
        if lo == hi:
            return s[lo]
        frac = idx - lo
        return s[lo] * (1 - frac) + s[hi] * frac

# ---------- Statistical: correlation & covariance ----------

class Corr:
    """CORR(x, y) — Pearson correlation."""
    def __init__(self): self.xs, self.ys = [], []
    def step(self, x, y):
        if x is not None and y is not None:
            self.xs.append(x); self.ys.append(y)
    def finalize(self):
        n = len(self.xs)
        if n < 2: return None
        mx, my = sum(self.xs)/n, sum(self.ys)/n
        cov = sum((a-mx)*(b-my) for a, b in zip(self.xs, self.ys)) / (n - 1)
        sx = math.sqrt(sum((a-mx)**2 for a in self.xs) / (n - 1))
        sy = math.sqrt(sum((b-my)**2 for b in self.ys) / (n - 1))
        return cov / (sx * sy) if sx and sy else None


class CovarSamp:
    """COVAR_SAMP(x, y) — sample covariance (ddof=1)."""
    def __init__(self): self.xs, self.ys = [], []
    def step(self, x, y):
        if x is not None and y is not None:
            self.xs.append(x); self.ys.append(y)
    def finalize(self):
        n = len(self.xs)
        if n < 2: return None
        mx, my = sum(self.xs)/n, sum(self.ys)/n
        return sum((a-mx)*(b-my) for a, b in zip(self.xs, self.ys)) / (n - 1)


class CovarPop:
    """COVAR_POP(x, y) — population covariance (ddof=0)."""
    def __init__(self): self.xs, self.ys = [], []
    def step(self, x, y):
        if x is not None and y is not None:
            self.xs.append(x); self.ys.append(y)
    def finalize(self):
        n = len(self.xs)
        if n == 0: return None
        mx, my = sum(self.xs)/n, sum(self.ys)/n
        return sum((a-mx)*(b-my) for a, b in zip(self.xs, self.ys)) / n


class Beta:
    """BETA(x, y) — slope of y on x (OLS)."""
    def __init__(self): self.xs, self.ys = [], []
    def step(self, x, y):
        if x is not None and y is not None:
            self.xs.append(x); self.ys.append(y)
    def finalize(self):
        n = len(self.xs)
        if n < 2: return None
        mx, my = sum(self.xs)/n, sum(self.ys)/n
        cov = sum((a-mx)*(b-my) for a, b in zip(self.xs, self.ys)) / (n - 1)
        var_x = sum((a-mx)**2 for a in self.xs) / (n - 1)
        return cov / var_x if var_x else None


# ---------- Statistical: ranking ----------

class RankPercentile:
    """RANK_PERCENTILE(x) — fraction of values ≤ x (empirical CDF)."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        if not self.v: return None
        s = sorted(self.v)
        n = len(s)
        import bisect
        last = s[-1]
        idx = bisect.bisect_right(s, last)
        return idx / n


# ---------- Statistical: rolling / window helpers ----------

class Product:
    """PRODUCT(x) — multiplicative product."""
    def __init__(self): self.p = 1.0; self.count = 0
    def step(self, x):
        if x is not None:
            self.p *= x
            self.count += 1
    def finalize(self):
        return self.p if self.count else None


class FirstNonNull:
    """FIRST_NON_NULL(x) — first non-null value encountered."""
    def __init__(self): self.v = None; self.set = False
    def step(self, x):
        if not self.set and x is not None:
            self.v = x
            self.set = True
    def finalize(self):
        return self.v


class LastNonNull:
    """LAST_NON_NULL(x) — last non-null value encountered."""
    def __init__(self): self.v = None
    def step(self, x):
        if x is not None: self.v = x
    def finalize(self):
        return self.v


# ---------- Statistical: cumulative ----------

class CumSum:
    """CUM_SUM(x) — running sum (order-sensitive; use with ORDER BY)."""
    def __init__(self): self.total = 0.0
    def step(self, x):
        if x is not None: self.total += x
    def finalize(self):
        return self.total


class CumMax:
    """CUM_MAX(x) — running maximum."""
    def __init__(self): self.mx = None
    def step(self, x):
        if x is not None:
            self.mx = x if self.mx is None else max(self.mx, x)
    def finalize(self):
        return self.mx


class CumMin:
    """CUM_MIN(x) — running minimum."""
    def __init__(self): self.mn = None
    def step(self, x):
        if x is not None:
            self.mn = x if self.mn is None else min(self.mn, x)
    def finalize(self):
        return self.mn


# ---------- Skewness & kurtosis ----------

class Skewness:
    """SKEWNESS(x) — sample skewness (Fisher)."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        n = len(self.v)
        if n < 3: return None
        m = sum(self.v) / n
        s = math.sqrt(sum((x - m) ** 2 for x in self.v) / (n - 1))
        if s == 0: return 0.0
        return (n / ((n - 1) * (n - 2))) * sum(((x - m) / s) ** 3 for x in self.v)


class Kurtosis:
    """KURTOSIS(x) — sample excess kurtosis (Fisher)."""
    def __init__(self): self.v = []
    def step(self, x):
        if x is not None: self.v.append(x)
    def finalize(self):
        n = len(self.v)
        if n < 4: return None
        m = sum(self.v) / n
        s2 = sum((x - m) ** 2 for x in self.v) / n
        if s2 == 0: return 0.0
        m4 = sum((x - m) ** 4 for x in self.v) / n
        return m4 / (s2 * s2) - 3.0


# ---------- String aggregations ----------

class GroupConcat:
    """GROUP_CONCAT(x, sep) — SQLite already has this; keep for parity."""
    def __init__(self, sep=','): self.sep = sep; self.v = []
    def step(self, x):
        if x is not None: self.v.append(str(x))
    def finalize(self):
        return self.sep.join(self.v) if self.v else None


# ============================================================
# REGISTRATION — attach to every new SQLite connection
# ============================================================

# 1. Snapshot original connect (unwrap if already patched)
_orig_connect = sqlite3.connect

# 2. Shared registration helper — the single source of truth
def _connect_rich(*args, **kwargs):
    conn = _orig_connect(*args, **kwargs)

    # ---- 1-arg aggregates ----
    conn.create_aggregate("median",           1, Median)
    conn.create_aggregate("mode",             1, Mode)
    conn.create_aggregate("variance",         1, Variance)
    conn.create_aggregate("var_samp",         1, Variance)       
    conn.create_aggregate("var_pop",          1, VarPop)
    conn.create_aggregate("stdev",            1, StDev)
    conn.create_aggregate("stddev",           1, StDev)          
    conn.create_aggregate("stddev_samp",      1, StDev)          
    conn.create_aggregate("stddev_pop",       1, StdDevPop)
    conn.create_aggregate("mad",              1, MAD)
    conn.create_aggregate("range",            1, Range)
    conn.create_aggregate("product",          1, Product)
    conn.create_aggregate("first_non_null",   1, FirstNonNull)
    conn.create_aggregate("last_non_null",    1, LastNonNull)
    conn.create_aggregate("cum_sum",          1, CumSum)
    conn.create_aggregate("cum_max",          1, CumMax)
    conn.create_aggregate("cum_min",          1, CumMin)
    conn.create_aggregate("skewness",         1, Skewness)
    conn.create_aggregate("kurtosis",         1, Kurtosis)
    conn.create_aggregate("rank_percentile",  1, RankPercentile)

    # ---- 2-arg aggregates ----
    conn.create_aggregate("corr",             2, Corr)
    conn.create_aggregate("covar_samp",       2, CovarSamp)
    conn.create_aggregate("covar_pop",        2, CovarPop)
    conn.create_aggregate("beta",             2, Beta)

    # ---- Variable-arg aggregates ----
    conn.create_aggregate("percentile_cont",  2, Percentile)    
    conn.create_aggregate("percentile_disc",  2, Percentile)    

    return conn

sqlite3.connect = _connect_rich

In [9]:
# 3. Hook SQLAlchemy so %sql also works
from sqlalchemy import event
from sqlalchemy.pool import Pool
import sqlite3

def _register_aggregates(conn):
    """Register all custom aggregates on a raw DBAPI connection."""
    conn.create_aggregate("median",           1, Median)
    conn.create_aggregate("mode",             1, Mode)
    conn.create_aggregate("variance",         1, Variance)
    conn.create_aggregate("var_samp",         1, Variance)
    conn.create_aggregate("var_pop",          1, VarPop)
    conn.create_aggregate("stdev",            1, StDev)
    conn.create_aggregate("stddev",           1, StDev)
    conn.create_aggregate("stddev_samp",      1, StDev)
    conn.create_aggregate("stddev_pop",       1, StdDevPop)
    conn.create_aggregate("mad",              1, MAD)
    conn.create_aggregate("range",            1, Range)
    conn.create_aggregate("product",          1, Product)
    conn.create_aggregate("first_non_null",   1, FirstNonNull)
    conn.create_aggregate("last_non_null",    1, LastNonNull)
    conn.create_aggregate("cum_sum",          1, CumSum)
    conn.create_aggregate("cum_max",          1, CumMax)
    conn.create_aggregate("cum_min",          1, CumMin)
    conn.create_aggregate("skewness",         1, Skewness)
    conn.create_aggregate("kurtosis",         1, Kurtosis)
    conn.create_aggregate("rank_percentile",  1, RankPercentile)
    conn.create_aggregate("corr",             2, Corr)
    conn.create_aggregate("covar_samp",       2, CovarSamp)
    conn.create_aggregate("covar_pop",        2, CovarPop)
    conn.create_aggregate("beta",             2, Beta)
    conn.create_aggregate("percentile_cont",  2, Percentile)
    conn.create_aggregate("percentile_disc",  2, Percentile)

# --- Hook SQLAlchemy pool so `%sql sqlite:///...` gets the same aggregates ---
@event.listens_for(Pool, "connect")
def _on_sa_connect(dbapi_conn, connection_record):
    _register_aggregates(dbapi_conn)

In [10]:
# Create (or connect to) a database file
conn = sqlite3.connect("stocks.db")

# Save the DataFrame as a table
df.to_sql("stock", conn, if_exists="replace", index=False)

conn.close()

In [11]:
%sql sqlite:///stocks.db

Question 1: What is the highest closing price ever?

Where the market was willing to pay the most — the peak of optimism.

In [12]:
# numpy
Close.max()

339.78692626953125

In [13]:
# pandas
df['Close'].max()

339.78692626953125

In [14]:
%%sql
SELECT max(Close)
FROM stock;

 * sqlite:///stocks.db
Done.


max(Close)
339.78692626953125


Question 2: What is the lowest closing price ever? 

Peak fear / capitulation point. Establishes a floor of despair. If the stock revisits it, market is questioning the company's survival.

In [15]:
# numpy
Close.min()

24.11261749267578

In [16]:
# pandas
df['Close'].min()

24.11261749267578

In [17]:
%%sql
Select min(Close)
from stock;

 * sqlite:///stocks.db
Done.


min(Close)
24.11261749267578


Question 3: On which date was the stock at its peak?

 When the crowd was most enthusiastic. Anchors behavioral memory — investors still holding from that date feel pain. This creates resistance.

In [18]:
# numpy
dates[np.argmax(Close)]

'2026-07-28 00:00:00-04:00'

In [19]:
# pandas
df.loc[df['Close'].idxmax()].Date

'2026-07-28 00:00:00-04:00'

In [20]:
%%sql
select Date
from stock
order by Close desc 
limit 1;

 * sqlite:///stocks.db
Done.


Date
2026-07-28 00:00:00-04:00


Question 4: What is the average closing price?
The "fair value" over the period

In [21]:
# numpy
Close.mean()

132.02770385392913

In [22]:
# pandas
df["Close"].mean()

132.02770385392913

In [23]:
%%sql
select avg(close)
from stock;

 * sqlite:///stocks.db
Done.


avg(close)
132.02770385392913


Question 5: What is the median closing price?

The "typical" price without outliers

In [24]:
# numpy
np.median(Close)

136.76878356933594

In [25]:
# pandas
df['Close'].median()

136.76878356933594

In [26]:
%%sql 
select avg(close) as median
from (
    select close 
    from stock 
    order by close 
    limit 2 - (select count(*) from stock) % 2 
    offset (select (count(*) - 1) / 2 from stock));

 * sqlite:///stocks.db
Done.


median
136.76878356933594


Question 6: What is the daily price range (High - Low)

Intraday volatility

In [27]:
# numpy
(High - Low).mean()

2.783437928362192

In [28]:
# pandas
(df['High'] - df['Low']).mean()

2.783437928362192

In [29]:
%%sql 
select avg(high - low) 
from stock

 * sqlite:///stocks.db
Done.


avg(high - low)
2.7834379283621895


Question 7: What is the intraday volatility (std of range)?

How unstable the daily battle is

In [30]:
# numpy
(High - Low).std(ddof=1)

2.3982272658597483

In [31]:
# pandas
(df['High'] - df['Low']).std()

2.3982272658597483

In [32]:
%%sql 
select stdev(high - low) as stdev 
from stock;

 * sqlite:///stocks.db
Done.


stdev
2.3982272658597483


Question 8: What is the open-close gap per day

Overnight sentiment shift

In [33]:
# numpy
Close - Open

array([ 0.63803323,  0.10062051,  0.69520278, ..., -8.33999634,
       -0.88000488, -0.1499939 ])

In [34]:
# pandas
df['Close'] - df['Open']

0       0.638033
1       0.100621
2       0.695203
3       0.391052
4      -0.045738
          ...   
2507   -1.910004
2508    3.339996
2509   -8.339996
2510   -0.880005
2511   -0.149994
Length: 2512, dtype: float64

In [35]:
%%sql 
select date, close - open as gap 
from stock 
limit 3;

 * sqlite:///stocks.db
Done.


Date,gap
2016-09-12 00:00:00-04:00,0.638033225745712
2016-09-13 00:00:00-04:00,0.10062050642081033
2016-09-14 00:00:00-04:00,0.6952027801270795


Question 9: What is the 52 week high?

Recent momentum ceiling

In [36]:
# numpy 
Close[-252:].max()

339.78692626953125

In [37]:
# pandas
df['Close'].tail(252).max()

339.78692626953125

In [38]:
%%sql 
select max(close) 
from (
    select close 
    from stock 
    order by date desc 
    limit 252
    );

 * sqlite:///stocks.db
Done.


max(close)
339.78692626953125


Question 10: What is the 52 week low?

Recent capitulation bottom

In [39]:
# numpy
Close[-252:].min()

225.95530700683597

In [40]:
# pandas
df['Close'].tail(252).min()

225.95530700683597

In [41]:
%%sql 
select min(close) 
from (
    select close 
    from stock 
    order by date desc 
    limit 252
    );

 * sqlite:///stocks.db
Done.


min(close)
225.95530700683597


Question 11: What is the price range over the period?

Total chaos vs stability over your window

In [42]:
# numpy
Close.max() - Close.min()

315.67430877685547

In [43]:
# pandas
df['Close'].max() - df['Close'].min()

315.67430877685547

In [44]:
%%sql
select max(Close) - min(Close) 
from stock;

 * sqlite:///stocks.db
Done.


max(Close) - min(Close)
315.67430877685547


Question 12: What is the mid-price (H+L)/2

Fair value intraday - where buyers and sellers meet

In [45]:
# numpy
(High + Low) / 2

array([ 23.81189528,  24.70148202,  25.34179779, ..., 323.39498901,
       317.80000305, 314.5249939 ])

In [46]:
# pandas
(df['High'] + df['Low']) / 2

0        23.811895
1        24.701482
2        25.341798
3        26.209663
4        26.318285
           ...    
2507    325.964996
2508    327.459991
2509    323.394989
2510    317.800003
2511    314.524994
Length: 2512, dtype: float64

In [47]:
%%sql 
select (High + Low) / 2 as MidPrice 
from stock 
limit 5;

 * sqlite:///stocks.db
Done.


MidPrice
23.811895279688507
24.701482024157983
25.341797785043113
26.20966310627172
26.31828524833001


Question 13: What is the total return over the period?

Bottom line: Did I make money?

In [48]:
# numpy
(Close[-1] / Close[0] - 1) * 100

1207.7800302421556

In [49]:
# pandas
(df['Close'].iloc[-1] / df['Close'].iloc[0] - 1) * 100

1207.7800302421556

In [50]:
%%sql 
select (last_close / first_close - 1) * 100 as total_return 
from (
    select first_value(Close) over (order by Date asc) as first_close,
    last_value(Close) over (
        order by Date asc
        rows between unbounded preceding and unbounded following
    ) as last_close
    from stock 
    limit 1
)

 * sqlite:///stocks.db
Done.


total_return
1207.7800302421556


Question 14: What is the daily return?

In [51]:
# numpy 
np.concatenate([[np.nan], np.diff(Close) / Close[:-1]])

array([        nan,  0.02380494,  0.03538659, ..., -0.02510585,
       -0.01171985, -0.00278289])

In [52]:
# pandas
df['Close'].pct_change()

0            NaN
1       0.023805
2       0.035387
3       0.033999
4      -0.005624
          ...   
2507   -0.000523
2508    0.010001
2509   -0.025106
2510   -0.011720
2511   -0.002783
Name: Close, Length: 2512, dtype: float64

In [53]:
%%sql 
select (close - lag(Close) over (order by Date)) / lag(Close) over (order by Date) as daily_return 
from stock 
limit 5;

 * sqlite:///stocks.db
Done.


daily_return
None
0.023804939678638337
0.03538659080729763
0.0339985323375995
-0.005624430908411218


Question 15: What is the average daily return?

In [54]:
# numpy 
np.nanmean(DailyReturn)

0.0011927380978349902

In [55]:
# pandas
df['Close'].pct_change().mean()

0.0011927380978349904

In [ ]:
%%sql 
create view if not exists returns as 
select Date, Close, Volume, (Close - lag(Close) over (order by Date)) / lag(Close) over (order by date) as DailyReturn from stock;

select avg(DailyReturn) from returns;

 * sqlite:///stocks.db
Done.
Done.


avg(DailyReturn)
0.0011927380978349871
